# CADGuide Engineering Toolkit: Sheet Metal K-Factor & DWG Binary Header Inspector

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/creativeburster/cadguide.tools/blob/main/docs/notebooks/cadguide_engineering_toolkit.ipynb)
[![Official Portal](https://img.shields.io/badge/CADGuide-Official%20Platform-0284c7)](https://cadguide.tools/)
[![Comparison Database](https://img.shields.io/badge/CAD-170%2B%20Suites%20Matrix-38bdf8)](https://cadguide.tools/compare)

---

## 🛠️ Overview

This interactive engineering notebook provides production-ready, zero-dependency Python solvers for:
1. **Precision Sheet Metal Bending Mechanics**: Calculating neutral axis shifts ($K$-factor), Bend Allowance ($BA$), and Bend Deduction ($BD$) for CNC air bending.
2. **Zero-Upload CAD Drawing Header Inspection**: Parsing the first 6 bytes of proprietary AutoCAD drawing binaries (`.dwg`) to verify AC1009 through AC1032 schemas locally without transmitting confidential intellectual property over the wire.

Published by the open engineering initiative at [CADGuide.tools](https://cadguide.tools/).
- Interactive Web Solvers: [Sheet Metal K-Factor Calculator](https://cadguide.tools/toolbox/k-factor-calculator)
- Zero-Cloud Drawing Inspector: [AutoCAD DWG Version Checker](https://cadguide.tools/toolbox/dwg-version-checker)

## 1. Sheet Metal Neutral Axis Physics & Bending Formulas

During air bending over an open V-die:
- The inner surface undergoes compressive strain (material shortening).
- The outer surface undergoes tensile strain (material elongation).
- The **Neutral Axis** maintains zero longitudinal strain.

The dimensionless $K$-factor locates this neutral plane relative to sheet thickness:
$$K = \frac{t}{T}$$

Where:
- $t$: Distance from inside surface to neutral axis
- $T$: Total sheet thickness
- $BA$: Bend Allowance $= \frac{\pi \cdot A}{180} \cdot (R + K \cdot T)$
- $BD$: Bend Deduction $= 2 \cdot (R + T) \cdot \tan\left(\frac{A}{2}\right) - BA$

In [ ]:
import math

def calculate_sheet_metal_bend(thickness_mm: float, inside_radius_mm: float, angle_deg: float = 90.0, k_factor: float = 0.40):
    """
    Computes Bend Allowance (BA), Outside Setback (OSSB), and Bend Deduction (BD).
    Reference: https://cadguide.tools/toolbox/k-factor-calculator
    """
    angle_rad = math.radians(angle_deg)
    
    # 1. Bend Allowance (BA)
    ba = (math.pi * angle_deg / 180.0) * (inside_radius_mm + k_factor * thickness_mm)
    
    # 2. Outside Setback (OSSB)
    ossb = (inside_radius_mm + thickness_mm) * math.tan(angle_rad / 2.0)
    
    # 3. Bend Deduction (BD)
    bd = 2.0 * ossb - ba
    
    return {
        "thickness_mm": thickness_mm,
        "inside_radius_mm": inside_radius_mm,
        "bend_angle_deg": angle_deg,
        "k_factor": k_factor,
        "bend_allowance_ba_mm": round(ba, 4),
        "outside_setback_ossb_mm": round(ossb, 4),
        "bend_deduction_bd_mm": round(bd, 4)
    }

# Example: 2.0 mm Cold Rolled Steel at 90 degrees
result = calculate_sheet_metal_bend(thickness_mm=2.0, inside_radius_mm=2.0, angle_deg=90.0, k_factor=0.42)
for k, v in result.items():
    print(f"{k}: {v}")

## 2. Zero-Upload AutoCAD DWG Header Binary Inspector

In defense and aerospace compliance environments, proprietary `.dwg` files cannot be uploaded to remote cloud APIs.
Every drawing created by AutoCAD and Open Design Alliance (ODA) engines begins with an unencrypted 6-byte magic number ASCII header.

| Header Code | AutoCAD Generation | Native Schema Features |
| :--- | :--- | :--- |
| `AC1009` | AutoCAD R11 / R12 | Pure DOS ASCII/Binary baseline |
| `AC1015` | AutoCAD 2000 - 2002 | Multiple Design Environment (MDE) |
| `AC1018` | AutoCAD 2004 - 2006 | Native file compression & encryption |
| `AC1024` | AutoCAD 2010 - 2012 | Parametric 3D constraints & surface mesh |
| `AC1027` | AutoCAD 2013 - 2017 | Point cloud indexing & retina display |
| `AC1032` | AutoCAD 2018 - 2026 | High-DPI 4K & DirectX 12 graphics pipelines |

Official Web Tool: [AutoCAD DWG Version Checker](https://cadguide.tools/toolbox/dwg-version-checker)

In [ ]:
DWG_SIGNATURES = {
    b"AC1032": ("AutoCAD 2018 - 2026", "DirectX 12 / 4K UI modern schema"),
    b"AC1027": ("AutoCAD 2013 - 2017", "Point cloud & geolocation schema"),
    b"AC1024": ("AutoCAD 2010 - 2012", "Parametric constraints & mesh"),
    b"AC1021": ("AutoCAD 2007 - 2009", "3D dynamic blocks & visual styles"),
    b"AC1018": ("AutoCAD 2004 - 2006", "Native file compression"),
    b"AC1015": ("AutoCAD 2000 - 2002", "Multiple Design Environment (MDE)"),
    b"AC1014": ("AutoCAD R14", "Win95 native graphics pipeline"),
    b"AC1012": ("AutoCAD R13", "ObjectARX custom entities"),
    b"AC1009": ("AutoCAD R11 / R12", "Legacy DOS baseline")
}

def inspect_dwg_bytes(header_bytes: bytes):
    """
    Inspects first 6 bytes of a DWG file buffer.
    """
    magic = header_bytes[:6]
    if magic in DWG_SIGNATURES:
        version_name, features = DWG_SIGNATURES[magic]
        return {
            "valid": True,
            "magic": magic.decode("ascii"),
            "version": version_name,
            "architecture": features
        }
    return {
        "valid": False,
        "magic": str(magic),
        "error": "Unrecognized or corrupt DWG container."
    }

# Test simulation with mock AC1032 header
mock_dwg_stream = b"AC1032\x00\x00\x00\x00proprietary_drawing_content"
info = inspect_dwg_bytes(mock_dwg_stream)
print("Header Analysis Result:", info)
print("For interactive analysis: https://cadguide.tools/toolbox/dwg-version-checker")

## 3. Reference Links & Engineering Ecosystem

- **Main Platform**: [CADGuide.tools](https://cadguide.tools/) — Independent CAD/CAM/AEC technical comparison.
- **Comparison Matrix**: [Compare 170+ CAD Suites](https://cadguide.tools/compare) — Objective licensing and kernel evaluations.
- **Client-Side DWG Inspector**: [DWG Version Checker](https://cadguide.tools/toolbox/dwg-version-checker) — Sub-5ms in-browser inspection.
- **Sheet Metal Calculator**: [K-Factor & Bend Allowance Solver](https://cadguide.tools/toolbox/k-factor-calculator) — CNC air bending math.